# Step 3: Forward Curve + Spread Foundation

This notebook extends base tables with structural market signals:
- C1–C2 spread
- Curve state (contango vs backwardation)
- Roll pressure proxies
- Liquidity confirmation
- Basic returns

**Rules:**
- Still NO WASDE data
- Still NO ML features
- Purely market-structure signals


In [1]:
import pandas as pd
import numpy as np


In [2]:
BASE_PATH = "/Users/aryansinha/Desktop/WASDA"

# Load base tables
corn  = pd.read_csv(f"{BASE_PATH}/processed/base_tables/corn_base.csv",  parse_dates=["date"])
soy   = pd.read_csv(f"{BASE_PATH}/processed/base_tables/soybean_base.csv", parse_dates=["date"])
wheat = pd.read_csv(f"{BASE_PATH}/processed/base_tables/wheat_base.csv", parse_dates=["date"])

print("Base tables loaded:")
print(f"Corn:  {corn.shape}")
print(f"Soy:   {soy.shape}")
print(f"Wheat: {wheat.shape}")


Base tables loaded:
Corn:  (1259, 7)
Soy:   (1260, 7)
Wheat: (1259, 7)


## STEP 3.3 — Compute Core Spread Features

For each commodity:
- `spread_c1_c2` = C1_BID - C2_BID
- `spread_pct` = spread_c1_c2 / C1_BID

**Interpretation:**
- `spread_c1_c2 > 0` → backwardation (front month premium)
- `spread_c1_c2 < 0` → contango (back month premium)


In [3]:
# CORN: Compute spread features
corn["spread_c1_c2"] = corn["c1_bid"] - corn["c2_bid"]
corn["spread_pct"] = corn["spread_c1_c2"] / (corn["c1_bid"] + 1e-9)  # Add small epsilon to avoid division by zero

print("Corn spread features computed")
print(corn[["date", "c1_bid", "c2_bid", "spread_c1_c2", "spread_pct"]].tail(10))


Corn spread features computed
           date  c1_bid  c2_bid  spread_c1_c2  spread_pct
1249 2025-11-25   423.5  437.75        -14.25   -0.033648
1250 2025-11-26   428.0  445.50        -17.50   -0.040888
1251 2025-11-28   426.5  447.50        -21.00   -0.049238
1252 2025-12-01   433.5  445.00        -11.50   -0.026528
1253 2025-12-02   436.5  449.50        -13.00   -0.029782
1254 2025-12-03   431.0  443.00        -12.00   -0.027842
1255 2025-12-04   435.0  446.25        -11.25   -0.025862
1256 2025-12-05   435.5  444.50         -9.00   -0.020666
1257 2025-12-08   434.0  443.50         -9.50   -0.021889
1258 2025-12-09   435.0  447.75        -12.75   -0.029310


In [4]:
# SOYBEAN: Compute spread features
soy["spread_c1_c2"] = soy["s1_bid"] - soy["s2_bid"]
soy["spread_pct"] = soy["spread_c1_c2"] / (soy["s1_bid"] + 1e-9)

print("Soybean spread features computed")
print(soy[["date", "s1_bid", "s2_bid", "spread_c1_c2", "spread_pct"]].tail(10))


Soybean spread features computed
           date   s1_bid   s2_bid  spread_c1_c2  spread_pct
1250 2025-11-26  1131.75  1141.00         -9.25   -0.008173
1251 2025-11-28  1137.25  1145.25         -8.00   -0.007035
1252 2025-12-01  1127.25  1137.50        -10.25   -0.009093
1253 2025-12-02  1124.00  1134.00        -10.00   -0.008897
1254 2025-12-03  1116.00  1126.50        -10.50   -0.009409
1255 2025-12-04  1119.50  1128.75         -9.25   -0.008263
1256 2025-12-05  1105.00  1115.50        -10.50   -0.009502
1257 2025-12-08  1093.25  1105.50        -12.25   -0.011205
1258 2025-12-09  1087.25  1098.00        -10.75   -0.009887
1259 2025-12-10  1084.25  1095.50        -11.25   -0.010376


In [5]:
# WHEAT: Compute spread features
wheat["spread_c1_c2"] = wheat["w1_bid"] - wheat["w2_bid"]
wheat["spread_pct"] = wheat["spread_c1_c2"] / (wheat["w1_bid"] + 1e-9)

print("Wheat spread features computed")
print(wheat[["date", "w1_bid", "w2_bid", "spread_c1_c2", "spread_pct"]].tail(10))


Wheat spread features computed
           date  w1_bid  w2_bid  spread_c1_c2  spread_pct
1249 2025-11-25  522.00  538.25        -16.25   -0.031130
1250 2025-11-26  522.00  540.75        -18.75   -0.035920
1251 2025-11-28  519.50  538.00        -18.50   -0.035611
1252 2025-12-01  515.25  535.00        -19.75   -0.038331
1253 2025-12-02  545.00  541.00          4.00    0.007339
1254 2025-12-03  515.25  538.25        -23.00   -0.044639
1255 2025-12-04  539.00  540.00         -1.00   -0.001855
1256 2025-12-05  515.25  535.50        -20.25   -0.039301
1257 2025-12-08  515.25  534.75        -19.50   -0.037846
1258 2025-12-09  515.25  534.00        -18.75   -0.036390


## STEP 3.4 — Roll / Positioning Proxies

Approximates:
- Money rolling out of front vs back
- Delivery stress near expiry


In [6]:
# CORN: Roll pressure proxies
corn["d_c1_oi"] = corn["c1_oi"].diff()
corn["d_c2_oi"] = corn["c2_oi"].diff()
corn["roll_pressure"] = corn["d_c1_oi"] - corn["d_c2_oi"]

print("Corn roll pressure computed")
print(corn[["date", "c1_oi", "c2_oi", "d_c1_oi", "d_c2_oi", "roll_pressure"]].tail(10))


Corn roll pressure computed
           date    c1_oi     c2_oi  d_c1_oi  d_c2_oi  roll_pressure
1249 2025-11-25  77870.0  716267.0 -76689.0  33318.0      -110007.0
1250 2025-11-26  13468.0  745324.0 -64402.0  29057.0       -93459.0
1251 2025-11-28   8066.0  739987.0  -5402.0  -5337.0          -65.0
1252 2025-12-01   5861.0  727008.0  -2205.0 -12979.0        10774.0
1253 2025-12-02   4358.0  738669.0  -1503.0  11661.0       -13164.0
1254 2025-12-03   3888.0  730114.0   -470.0  -8555.0         8085.0
1255 2025-12-04   3395.0  730678.0   -493.0    564.0        -1057.0
1256 2025-12-05   2517.0  727826.0   -878.0  -2852.0         1974.0
1257 2025-12-08   1423.0  724884.0  -1094.0  -2942.0         1848.0
1258 2025-12-09      NaN       NaN      NaN      NaN            NaN


In [7]:
# SOYBEAN: Roll pressure proxies
soy["d_c1_oi"] = soy["s1_oi"].diff()
soy["d_c2_oi"] = soy["s2_oi"].diff()
soy["roll_pressure"] = soy["d_c1_oi"] - soy["d_c2_oi"]

print("Soybean roll pressure computed")
print(soy[["date", "s1_oi", "s2_oi", "d_c1_oi", "d_c2_oi", "roll_pressure"]].tail(10))


Soybean roll pressure computed
           date     s1_oi     s2_oi  d_c1_oi  d_c2_oi  roll_pressure
1250 2025-11-26  354944.0  251286.0  -1184.0   4144.0        -5328.0
1251 2025-11-28  354968.0  253194.0     24.0   1908.0        -1884.0
1252 2025-12-01  347646.0  255573.0  -7322.0   2379.0        -9701.0
1253 2025-12-02  338780.0  260833.0  -8866.0   5260.0       -14126.0
1254 2025-12-03  335615.0  268406.0  -3165.0   7573.0       -10738.0
1255 2025-12-04  329565.0  272604.0  -6050.0   4198.0       -10248.0
1256 2025-12-05  313054.0  273635.0 -16511.0   1031.0       -17542.0
1257 2025-12-08  295335.0  276326.0 -17719.0   2691.0       -20410.0
1258 2025-12-09       NaN       NaN      NaN      NaN            NaN
1259 2025-12-10       NaN       NaN      NaN      NaN            NaN


In [8]:
# WHEAT: Roll pressure proxies
wheat["d_c1_oi"] = wheat["w1_oi"].diff()
wheat["d_c2_oi"] = wheat["w2_oi"].diff()
wheat["roll_pressure"] = wheat["d_c1_oi"] - wheat["d_c2_oi"]

print("Wheat roll pressure computed")
print(wheat[["date", "w1_oi", "w2_oi", "d_c1_oi", "d_c2_oi", "roll_pressure"]].tail(10))


Wheat roll pressure computed
           date    w1_oi     w2_oi  d_c1_oi  d_c2_oi  roll_pressure
1249 2025-11-25  12283.0  232613.0 -13842.0   3675.0       -17517.0
1250 2025-11-26   1970.0  236480.0 -10313.0   3867.0       -14180.0
1251 2025-11-28    833.0  238097.0  -1137.0   1617.0        -2754.0
1252 2025-12-01    507.0  237289.0   -326.0   -808.0          482.0
1253 2025-12-02    347.0  239134.0   -160.0   1845.0        -2005.0
1254 2025-12-03    230.0  236999.0   -117.0  -2135.0         2018.0
1255 2025-12-04    163.0  241669.0    -67.0   4670.0        -4737.0
1256 2025-12-05    111.0  242978.0    -52.0   1309.0        -1361.0
1257 2025-12-08     87.0  244072.0    -24.0   1094.0        -1118.0
1258 2025-12-09      NaN       NaN      NaN      NaN            NaN


## STEP 3.5 — Liquidity Confirmation

Volume and open interest ratios to assess market depth


In [9]:
# CORN: Liquidity ratios
corn["vol_ratio"] = corn["c1_vol"] / (corn["c2_vol"] + 1e-9)
corn["oi_ratio"] = corn["c1_oi"] / (corn["c2_oi"] + 1e-9)

print("Corn liquidity ratios computed")
print(corn[["date", "c1_vol", "c2_vol", "vol_ratio", "c1_oi", "c2_oi", "oi_ratio"]].tail(10))


Corn liquidity ratios computed
           date    c1_vol    c2_vol  vol_ratio    c1_oi     c2_oi  oi_ratio
1249 2025-11-25  227795.0  281855.0   0.808199  77870.0  716267.0  0.108716
1250 2025-11-26  158325.0  310045.0   0.510652  13468.0  745324.0  0.018070
1251 2025-11-28    8642.0   87644.0   0.098603   8066.0  739987.0  0.010900
1252 2025-12-01    3265.0  168991.0   0.019321   5861.0  727008.0  0.008062
1253 2025-12-02    2267.0  211525.0   0.010717   4358.0  738669.0  0.005900
1254 2025-12-03    1340.0  187591.0   0.007143   3888.0  730114.0  0.005325
1255 2025-12-04    1242.0  151430.0   0.008202   3395.0  730678.0  0.004646
1256 2025-12-05    1038.0  112552.0   0.009222   2517.0  727826.0  0.003458
1257 2025-12-08    1738.0  114701.0   0.015152   1423.0  724884.0  0.001963
1258 2025-12-09     873.0  168071.0   0.005194      NaN       NaN       NaN


In [10]:
# SOYBEAN: Liquidity ratios
soy["vol_ratio"] = soy["s1_vol"] / (soy["s2_vol"] + 1e-9)
soy["oi_ratio"] = soy["s1_oi"] / (soy["s2_oi"] + 1e-9)

print("Soybean liquidity ratios computed")
print(soy[["date", "s1_vol", "s2_vol", "vol_ratio", "s1_oi", "s2_oi", "oi_ratio"]].tail(10))


Soybean liquidity ratios computed
           date    s1_vol    s2_vol  vol_ratio     s1_oi     s2_oi  oi_ratio
1250 2025-11-26  104778.0   52584.0   1.992583  354944.0  251286.0  1.412510
1251 2025-11-28   38009.0   20410.0   1.862273  354968.0  253194.0  1.401961
1252 2025-12-01  117117.0   63918.0   1.832301  347646.0  255573.0  1.360261
1253 2025-12-02  134455.0   79658.0   1.687903  338780.0  260833.0  1.298839
1254 2025-12-03  127538.0   70637.0   1.805541  335615.0  268406.0  1.250401
1255 2025-12-04  134989.0   76860.0   1.756297  329565.0  272604.0  1.208951
1256 2025-12-05  140332.0   83293.0   1.684799  313054.0  273635.0  1.144057
1257 2025-12-08  170570.0  112901.0   1.510793  295335.0  276326.0  1.068792
1258 2025-12-09  161540.0  116544.0   1.386086       NaN       NaN       NaN
1259 2025-12-10    6229.0    3148.0   1.978717       NaN       NaN       NaN


In [11]:
# WHEAT: Liquidity ratios
wheat["vol_ratio"] = wheat["w1_vol"] / (wheat["w2_vol"] + 1e-9)
wheat["oi_ratio"] = wheat["w1_oi"] / (wheat["w2_oi"] + 1e-9)

print("Wheat liquidity ratios computed")
print(wheat[["date", "w1_vol", "w2_vol", "vol_ratio", "w1_oi", "w2_oi", "oi_ratio"]].tail(10))


Wheat liquidity ratios computed
           date   w1_vol   w2_vol  vol_ratio    w1_oi     w2_oi  oi_ratio
1249 2025-11-25  26212.0  69580.0   0.376717  12283.0  232613.0  0.052804
1250 2025-11-26  18026.0  72214.0   0.249619   1970.0  236480.0  0.008331
1251 2025-11-28   1198.0  23906.0   0.050113    833.0  238097.0  0.003499
1252 2025-12-01    426.0  61051.0   0.006978    507.0  237289.0  0.002137
1253 2025-12-02    184.0  75307.0   0.002443    347.0  239134.0  0.001451
1254 2025-12-03    139.0  46731.0   0.002974    230.0  236999.0  0.000970
1255 2025-12-04     73.0  57135.0   0.001278    163.0  241669.0  0.000674
1256 2025-12-05     56.0  48904.0   0.001145    111.0  242978.0  0.000457
1257 2025-12-08     31.0  49765.0   0.000623     87.0  244072.0  0.000356
1258 2025-12-09     14.0  52441.0   0.000267      NaN       NaN       NaN


## STEP 3.6 — Basic Returns

Returns are market facts, not ML features yet.


In [12]:
# CORN: Basic returns
corn["ret_1d"] = corn["c1_bid"].pct_change(1)
corn["ret_5d"] = corn["c1_bid"].pct_change(5)
corn["ret_21d"] = corn["c1_bid"].pct_change(21)

print("Corn returns computed")
print(corn[["date", "c1_bid", "ret_1d", "ret_5d", "ret_21d"]].tail(10))


Corn returns computed
           date  c1_bid    ret_1d    ret_5d   ret_21d
1249 2025-11-25   423.5 -0.001179 -0.027555 -0.010514
1250 2025-11-26   428.0  0.010626 -0.004072 -0.007536
1251 2025-11-28   426.5 -0.003505  0.000587 -0.017847
1252 2025-12-01   433.5  0.016413  0.018203  0.009901
1253 2025-12-02   436.5  0.006920  0.029481  0.013937
1254 2025-12-03   431.0 -0.012600  0.017710 -0.008055
1255 2025-12-04   435.0  0.009281  0.016355  0.009867
1256 2025-12-05   435.5  0.001149  0.021102  0.001725
1257 2025-12-08   434.0 -0.003444  0.001153  0.012245
1258 2025-12-09   435.0  0.002304 -0.003436  0.019332


/var/folders/3j/0xbmd_c918x585z1bkf86vtw0000gn/T/ipykernel_34875/1525368772.py:2: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  corn["ret_1d"] = corn["c1_bid"].pct_change(1)
/var/folders/3j/0xbmd_c918x585z1bkf86vtw0000gn/T/ipykernel_34875/1525368772.py:3: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  corn["ret_5d"] = corn["c1_bid"].pct_change(5)
/var/folders/3j/0xbmd_c918x585z1bkf86vtw0000gn/T/ipykernel_34875/1525368772.py:4: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to c

In [13]:
# SOYBEAN: Basic returns
soy["ret_1d"] = soy["s1_bid"].pct_change(1)
soy["ret_5d"] = soy["s1_bid"].pct_change(5)
soy["ret_21d"] = soy["s1_bid"].pct_change(21)

print("Soybean returns computed")
print(soy[["date", "s1_bid", "ret_1d", "ret_5d", "ret_21d"]].tail(10))


Soybean returns computed
           date   s1_bid    ret_1d    ret_5d   ret_21d
1250 2025-11-26  1131.75  0.006895 -0.002863  0.051812
1251 2025-11-28  1137.25  0.004860  0.012689  0.056188
1252 2025-12-01  1127.25 -0.008793  0.001555  0.038940
1253 2025-12-02  1124.00 -0.002883  0.002900  0.025547
1254 2025-12-03  1116.00 -0.007117 -0.007117  0.009955
1255 2025-12-04  1119.50  0.003136 -0.010824  0.027064
1256 2025-12-05  1105.00 -0.012952 -0.028358 -0.008969
1257 2025-12-08  1093.25 -0.010633 -0.030162  0.008301
1258 2025-12-09  1087.25 -0.005488 -0.032696 -0.005488
1259 2025-12-10  1084.25 -0.002759 -0.028450 -0.008232


/var/folders/3j/0xbmd_c918x585z1bkf86vtw0000gn/T/ipykernel_34875/2610971123.py:2: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  soy["ret_1d"] = soy["s1_bid"].pct_change(1)
/var/folders/3j/0xbmd_c918x585z1bkf86vtw0000gn/T/ipykernel_34875/2610971123.py:3: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  soy["ret_5d"] = soy["s1_bid"].pct_change(5)
/var/folders/3j/0xbmd_c918x585z1bkf86vtw0000gn/T/ipykernel_34875/2610971123.py:4: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calli

In [14]:
# WHEAT: Basic returns
wheat["ret_1d"] = wheat["w1_bid"].pct_change(1)
wheat["ret_5d"] = wheat["w1_bid"].pct_change(5)
wheat["ret_21d"] = wheat["w1_bid"].pct_change(21)

print("Wheat returns computed")
print(wheat[["date", "w1_bid", "ret_1d", "ret_5d", "ret_21d"]].tail(10))


Wheat returns computed
           date  w1_bid    ret_1d    ret_5d   ret_21d
1249 2025-11-25  522.00 -0.000957 -0.044831 -0.006660
1250 2025-11-26  522.00  0.000000 -0.027933 -0.015094
1251 2025-11-28  519.50 -0.004789 -0.015632 -0.025785
1252 2025-12-01  515.25 -0.008181 -0.026912 -0.017636
1253 2025-12-02  545.00  0.057739  0.043062  0.022035
1254 2025-12-03  515.25 -0.054587 -0.012931 -0.053719
1255 2025-12-04  539.00  0.046094  0.032567 -0.020000
1256 2025-12-05  515.25 -0.044063 -0.008181 -0.069946
1257 2025-12-08  515.25  0.000000  0.000000 -0.038264
1258 2025-12-09  515.25  0.000000 -0.054587 -0.022296


/var/folders/3j/0xbmd_c918x585z1bkf86vtw0000gn/T/ipykernel_34875/1630989270.py:2: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  wheat["ret_1d"] = wheat["w1_bid"].pct_change(1)
/var/folders/3j/0xbmd_c918x585z1bkf86vtw0000gn/T/ipykernel_34875/1630989270.py:3: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  wheat["ret_5d"] = wheat["w1_bid"].pct_change(5)
/var/folders/3j/0xbmd_c918x585z1bkf86vtw0000gn/T/ipykernel_34875/1630989270.py:4: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior 

## STEP 3.7 — Sanity Checks

Check that spreads flip between backwardation and contango over time.


In [15]:
# CORN: Sanity checks
print("=== CORN SANITY CHECKS ===")
print("\nSpread statistics:")
print(corn["spread_c1_c2"].describe())
print(f"\nBackwardation days (spread > 0): {(corn['spread_c1_c2'] > 0).sum()}")
print(f"Contango days (spread < 0): {(corn['spread_c1_c2'] < 0).sum()}")
print(f"Zero spread days: {(corn['spread_c1_c2'] == 0).sum()}")
print("\nRecent spread behavior:")
print(corn[["date", "c1_bid", "c2_bid", "spread_c1_c2", "roll_pressure"]].tail(20))


=== CORN SANITY CHECKS ===

Spread statistics:
count    1228.000000
mean        2.055171
std        29.392807
min      -127.000000
25%       -13.500000
50%        -6.000000
75%         4.250000
max       138.000000
Name: spread_c1_c2, dtype: float64

Backwardation days (spread > 0): 431
Contango days (spread < 0): 777
Zero spread days: 20

Recent spread behavior:
           date  c1_bid  c2_bid  spread_c1_c2  roll_pressure
1239 2025-11-11  431.50  446.25        -14.75       -47610.0
1240 2025-11-12  435.00  449.00        -14.00       -28344.0
1241 2025-11-13  442.00  455.00        -13.00       -84429.0
1242 2025-11-14  430.00  443.50        -13.50       -48366.0
1243 2025-11-17  434.75  447.75        -13.00       -24305.0
1244 2025-11-18  435.50  448.75        -13.25       -23462.0
1245 2025-11-19  429.75  442.00        -12.25       -43077.0
1246 2025-11-20  426.25  437.75        -11.50       -57944.0
1247 2025-11-21  425.75  437.50        -11.75      -112274.0
1248 2025-11-24  424.00 

In [16]:
# SOYBEAN: Sanity checks
print("=== SOYBEAN SANITY CHECKS ===")
print("\nSpread statistics:")
print(soy["spread_c1_c2"].describe())
print(f"\nBackwardation days (spread > 0): {(soy['spread_c1_c2'] > 0).sum()}")
print(f"Contango days (spread < 0): {(soy['spread_c1_c2'] < 0).sum()}")
print(f"Zero spread days: {(soy['spread_c1_c2'] == 0).sum()}")
print("\nRecent spread behavior:")
print(soy[["date", "s1_bid", "s2_bid", "spread_c1_c2", "roll_pressure"]].tail(20))


=== SOYBEAN SANITY CHECKS ===

Spread statistics:
count    1195.000000
mean        5.487029
std        37.863609
min      -179.250000
25%       -15.000000
50%        -5.500000
75%        15.000000
max       173.000000
Name: spread_c1_c2, dtype: float64

Backwardation days (spread > 0): 477
Contango days (spread < 0): 711
Zero spread days: 7

Recent spread behavior:
           date   s1_bid   s2_bid  spread_c1_c2  roll_pressure
1240 2025-11-12  1108.25  1135.00        -26.75        -3475.0
1241 2025-11-13  1017.50  1145.25       -127.75        -6249.0
1242 2025-11-14      NaN  1122.50           NaN        -5074.0
1243 2025-11-17  1157.00  1163.25         -6.25       541774.0
1244 2025-11-18  1150.00  1157.00         -7.00        -5700.0
1245 2025-11-19  1135.00  1143.25         -8.25        -7077.0
1246 2025-11-20  1123.00  1132.75         -9.75        -9777.0
1247 2025-11-21  1125.50  1134.75         -9.25       -12324.0
1248 2025-11-24  1120.75  1129.50         -8.75         1337.0
12

In [17]:
# WHEAT: Sanity checks
print("=== WHEAT SANITY CHECKS ===")
print("\nSpread statistics:")
print(wheat["spread_c1_c2"].describe())
print(f"\nBackwardation days (spread > 0): {(wheat['spread_c1_c2'] > 0).sum()}")
print(f"Contango days (spread < 0): {(wheat['spread_c1_c2'] < 0).sum()}")
print(f"Zero spread days: {(wheat['spread_c1_c2'] == 0).sum()}")
print("\nRecent spread behavior:")
print(wheat[["date", "w1_bid", "w2_bid", "spread_c1_c2", "roll_pressure"]].tail(20))


=== WHEAT SANITY CHECKS ===

Spread statistics:
count    1126.000000
mean      -17.222913
std        36.556225
min      -547.250000
25%       -20.000000
50%       -12.750000
75%        -5.750000
max        28.000000
Name: spread_c1_c2, dtype: float64

Backwardation days (spread > 0): 134
Contango days (spread < 0): 986
Zero spread days: 6

Recent spread behavior:
           date  w1_bid  w2_bid  spread_c1_c2  roll_pressure
1239 2025-11-11  536.00  551.00        -15.00       -32796.0
1240 2025-11-12  536.50  553.00        -16.50       -32894.0
1241 2025-11-13  535.25  552.00        -16.75       -30225.0
1242 2025-11-14  526.50  541.00        -14.50       -17557.0
1243 2025-11-17  544.75  556.75        -12.00       -21890.0
1244 2025-11-18  546.50  558.50        -12.00        -3758.0
1245 2025-11-19  537.00  550.25        -13.25       -10190.0
1246 2025-11-20  527.75  541.75        -14.00       -11842.0
1247 2025-11-21  529.50  542.00        -12.50       -16283.0
1248 2025-11-24  522.50 

## STEP 3.8 — Save Outputs

Save extended tables with all curve and spread features.


In [18]:
# Save all curve tables
corn.to_csv(f"{BASE_PATH}/processed/curve_tables/corn_curve.csv", index=False)
soy.to_csv(f"{BASE_PATH}/processed/curve_tables/soy_curve.csv", index=False)
wheat.to_csv(f"{BASE_PATH}/processed/curve_tables/wheat_curve.csv", index=False)

print("✅ All curve tables saved:")
print(f"  - {BASE_PATH}/processed/curve_tables/corn_curve.csv")
print(f"  - {BASE_PATH}/processed/curve_tables/soy_curve.csv")
print(f"  - {BASE_PATH}/processed/curve_tables/wheat_curve.csv")
print("\nFinal shapes:")
print(f"  Corn:  {corn.shape}")
print(f"  Soy:   {soy.shape}")
print(f"  Wheat: {wheat.shape}")
print("\nNew columns added:")
print(f"  - spread_c1_c2, spread_pct")
print(f"  - d_c1_oi, d_c2_oi, roll_pressure")
print(f"  - vol_ratio, oi_ratio")
print(f"  - ret_1d, ret_5d, ret_21d")


✅ All curve tables saved:
  - /Users/aryansinha/Desktop/WASDA/processed/curve_tables/corn_curve.csv
  - /Users/aryansinha/Desktop/WASDA/processed/curve_tables/soy_curve.csv
  - /Users/aryansinha/Desktop/WASDA/processed/curve_tables/wheat_curve.csv

Final shapes:
  Corn:  (1259, 17)
  Soy:   (1260, 17)
  Wheat: (1259, 17)

New columns added:
  - spread_c1_c2, spread_pct
  - d_c1_oi, d_c2_oi, roll_pressure
  - vol_ratio, oi_ratio
  - ret_1d, ret_5d, ret_21d


## Summary

All three commodities now have:
- ✅ Spread features (spread_c1_c2, spread_pct)
- ✅ Roll pressure proxies (d_c1_oi, d_c2_oi, roll_pressure)
- ✅ Liquidity confirmation (vol_ratio, oi_ratio)
- ✅ Basic returns (ret_1d, ret_5d, ret_21d)

**Next steps:**
- These tables are ready for WASDE event alignment (Step 4)
- Still no ML features - these are pure market structure signals
